# Retrievers in LangChain

# 1. Introduction

A **Retriever** in LangChain is a component responsible for retrieving relevant documents or document chunks based on a user's query.

The main purpose of a Retriever is:

> Given a query, find the most relevant pieces of information from a knowledge source.

A Retriever is an important component of:

- RAG (Retrieval-Augmented Generation)
- Question answering
- Semantic search
- Knowledge-base systems
- Document search
- Agentic applications

The basic idea is:

    User Query
         ↓
      Retriever
         ↓
    Relevant Documents
         ↓
        LLM
         ↓
       Answer

---

# 2. Why Do We Need Retrievers?

An LLM does not necessarily have access to your private documents.

Suppose you have:

    company_policy.pdf
    employee_handbook.pdf
    security_policy.pdf

The user asks:

    "How many annual leave days are employees allowed?"

The LLM cannot reliably answer from those private documents unless the relevant information is provided to it.

A Retriever searches the knowledge base:

    User Query
         ↓
    Retriever
         ↓
    Relevant Chunks
         ↓
    Prompt
         ↓
    LLM
         ↓
    Answer

This is the core idea behind RAG.

---

# 3. Retriever in RAG

A typical RAG pipeline is:

    Documents
        ↓
    Document Loader
        ↓
    Text Splitter
        ↓
    Chunks
        ↓
    Embedding Model
        ↓
    Vector Store
        ↓
    Retriever
        ↓
    Relevant Chunks
        ↓
    Prompt
        ↓
    LLM
        ↓
    Answer

The Retriever sits between the knowledge store and the LLM.

---

# 4. What Does a Retriever Do?

A Retriever generally performs this process:

    Query
      ↓
    Search Knowledge Source
      ↓
    Rank / Select Relevant Documents
      ↓
    Return Documents

For example:

    Query:
    "What is chunk overlap?"

The Retriever might return:

    Document 1:
    "Chunk overlap preserves context..."

    Document 2:
    "Overlap specifies the shared content..."

    Document 3:
    "Chunking strategies..."

The LLM can then use these documents to generate an answer.

---

# 5. Retriever Input and Output

A Retriever typically follows a simple interface:

    Query
      ↓
    Retriever
      ↓
    List of Documents

Input:

    "What is LangChain?"

Output conceptually:

    [
        Document(...),
        Document(...),
        Document(...)
    ]

The returned objects are usually LangChain `Document` objects.

---

# 6. Retriever vs Vector Store

This is one of the most important concepts.

A **Vector Store** is responsible for storing and searching vectors.

A **Retriever** is responsible for retrieving relevant documents.

Conceptually:

    Vector Store
         ↓
    Search vectors
         ↓
    Retriever
         ↓
    Relevant Documents

A Vector Store can often be converted into a Retriever.

Example:

    retriever = vector_store.as_retriever()

Then:

    retriever.invoke("What is LangChain?")

---

# 7. Vector Store Responsibility

A Vector Store generally handles:

- Vector storage
- Similarity search
- Metadata
- Indexing
- Vector retrieval

Conceptually:

    Documents
       ↓
    Embeddings
       ↓
    Vector Store
       ↓
    Similarity Search

---

# 8. Retriever Responsibility

A Retriever provides a higher-level retrieval interface.

Conceptually:

    User Query
        ↓
    Retriever
        ↓
    Relevant Documents

The Retriever abstracts away some of the details of the underlying search mechanism.

---

# 9. Easy Difference

Remember:

    Vector Store
    → Where vectors are stored and searched

    Retriever
    → How relevant documents are retrieved

Or:

    Vector Store = Storage + Search

    Retriever = Retrieval Interface

---

# 10. Retriever as a Runnable

A very important LangChain concept is that a Retriever implements the Runnable interface.

Therefore, you can use:

    retriever.invoke(query)

You can also use Runnable composition.

For example:

    chain = retriever | ...

This allows Retrievers to participate in LCEL pipelines.

---

# 11. Basic Retriever Example

Suppose we already have a vector store:

    vector_store = ...

We can create a Retriever:

    retriever = vector_store.as_retriever()

Then:

    documents = retriever.invoke(
        "What is LangChain?"
    )

Now `documents` contains relevant Document objects.

---

# 12. Retriever Architecture

    User Query
         ↓
    ┌────────────┐
    │ Retriever  │
    └─────┬──────┘
          ↓
    Search Knowledge Base
          ↓
    Rank / Select Results
          ↓
    Relevant Documents
          ↓
         LLM

---

# 13. Types of Retrievers

There are many retrieval strategies.

Important ones include:

1. Vector Store Retriever
2. Similarity Retriever
3. MMR Retriever
4. Multi-Query Retriever
5. Contextual Compression Retriever
6. Parent Document Retriever
7. Self-Query Retriever
8. Time-Weighted Retriever
9. Ensemble Retriever
10. BM25 / Keyword Retriever
11. Multi-Vector Retriever

Different retrievers solve different retrieval problems.

---

# 14. Vector Store Retriever

The most common starting point is a Retriever created from a Vector Store.

Example:

    retriever = vector_store.as_retriever()

Query:

    documents = retriever.invoke(
        "What are Runnables?"
    )

Architecture:

    Query
      ↓
    Retriever
      ↓
    Vector Store
      ↓
    Similarity Search
      ↓
    Documents

---

# 15. Search Type

When converting a Vector Store into a Retriever, you can often specify the retrieval strategy.

Common search types include:

    similarity
    mmr
    similarity_score_threshold

Example:

    retriever = vector_store.as_retriever(
        search_type="similarity"
    )

The exact supported search types depend on the Vector Store integration.

---

# 16. Similarity Search Retriever

Similarity search retrieves documents that are most similar to the query.

Conceptually:

    Query
      ↓
    Query Embedding
      ↓
    Vector Search
      ↓
    Most Similar Documents

Example:

    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 3
        }
    )

Here:

    k = 3

means the Retriever attempts to return three results.

---

# 17. Top-K Retrieval

`k` represents the number of results to retrieve.

Example:

    k = 5

means:

    Query
      ↓
    Search
      ↓
    Top 5 Documents

Conceptually:

    Result 1 → High relevance
    Result 2 → High relevance
    Result 3 → Relevant
    Result 4 → Relevant
    Result 5 → Relevant

---

# 18. Why Not Retrieve Everything?

Suppose the database contains:

    1,000,000 chunks

The user asks one question.

Returning all one million chunks would be:

- Expensive
- Slow
- Noisy
- Impossible to fit into a normal LLM context window

Instead:

    1,000,000 chunks
           ↓
        Retriever
           ↓
       Top 5 chunks
           ↓
          LLM

Retrieval reduces the amount of information sent to the LLM.

---

# 19. Similarity Score Threshold

Another retrieval strategy is to return documents only when their similarity is sufficiently high.

Conceptually:

    Query
      ↓
    Similarity Search
      ↓
    Score
      ↓
    Is score good enough?
      ↓
    Relevant Documents

Example:

    retriever = vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "score_threshold": 0.7
        }
    )

The exact meaning and range of the score depends on the vector store and distance/similarity implementation.

Always verify how the backend defines its score.

---

# 20. Why Use a Threshold?

Suppose the query is:

    "Explain quantum computing."

But your database contains only:

    Python tutorials
    SQL notes
    LangChain documentation

The Retriever may find the "least unrelated" documents even though none are actually relevant.

A threshold can help avoid returning weak matches.

Conceptually:

    High relevance
       ↓
    Retrieve

    Low relevance
       ↓
    Ignore

---

# 21. Maximum Marginal Relevance (MMR)

MMR stands for:

> Maximum Marginal Relevance

MMR attempts to balance:

1. Relevance to the query
2. Diversity among retrieved documents

This is useful when similarity search returns highly repetitive chunks.

---

# 22. Problem with Pure Similarity Search

Suppose the top results are:

    Chunk A → about Python functions
    Chunk B → almost identical to A
    Chunk C → almost identical to A
    Chunk D → about Python classes

Pure similarity search may return A, B, and C.

But they provide nearly identical information.

MMR tries to return:

    Chunk A → highly relevant
    Chunk D → relevant but different
    Chunk E → another useful perspective

This can provide more diverse context.

---

# 23. MMR Architecture

    Query
      ↓
    Candidate Documents
      ↓
    Relevance + Diversity
      ↓
    MMR Selection
      ↓
    Diverse Relevant Documents

---

# 24. MMR Example

Conceptually:

    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 4,
            "fetch_k": 20
        }
    )

Here:

    fetch_k
    ↓
    Initial candidate pool

    k
    ↓
    Final number of selected documents

The exact behavior depends on the vector store implementation.

---

# 25. `k` vs `fetch_k`

This is important.

### `fetch_k`

Number of candidate documents considered before final selection.

### `k`

Number of documents ultimately returned.

Example:

    fetch_k = 20
    k = 4

Conceptually:

    Query
      ↓
    Fetch 20 candidates
      ↓
    MMR
      ↓
    Select best 4
      ↓
    Return

---

# 26. Multi-Query Retriever

Sometimes a user's query can be ambiguous or phrased in a way that does not match the wording of the documents.

A **Multi-Query Retriever** uses an LLM to generate multiple alternative queries.

Example:

    Original Query:
    "How does LangChain execute multiple operations?"

The system may generate:

    Query 1:
    "What is RunnableParallel?"

    Query 2:
    "How can multiple Runnables execute in parallel?"

    Query 3:
    "How does LangChain support parallel Runnable execution?"

Then each query can be used for retrieval.

---

# 27. Multi-Query Architecture

    User Query
         ↓
        LLM
         ↓
    ┌────┼────┐
    ↓    ↓    ↓
    Q1   Q2   Q3
    ↓    ↓    ↓
    Search each query
    └────┼────┘
         ↓
    Combine Results
         ↓
    Remove Duplicates
         ↓
    Relevant Documents

This can improve recall.

---

# 28. Why Multi-Query Helps

A single query may fail because of wording differences.

For example:

    User:
    "How do I make several LangChain operations happen together?"

Document:

    "RunnableParallel executes multiple Runnables concurrently."

The words differ significantly.

Alternative queries can bridge this gap.

---

# 29. Contextual Compression Retriever

Sometimes retrieved documents contain the answer but also contain a lot of unnecessary information.

Example:

    Retrieved Document
    ┌──────────────────────────────┐
    │ Relevant information          │
    │ Irrelevant information        │
    │ More irrelevant information   │
    │ Relevant information          │
    │ Unrelated section             │
    └──────────────────────────────┘

A **Contextual Compression Retriever** can compress the retrieved documents to retain only information relevant to the query.

---

# 30. Contextual Compression Architecture

    Query
      ↓
    Base Retriever
      ↓
    Large Documents
      ↓
    Compressor
      ↓
    Relevant Content
      ↓
    LLM

This can reduce unnecessary context.

---

# 31. Parent Document Retriever

Small chunks can improve retrieval precision, but they may lack context.

Large chunks provide more context, but retrieval may become less precise.

The **Parent Document Retriever** attempts to balance these trade-offs.

Conceptually:

    Parent Document
          ↓
    ┌─────┼─────┐
    ↓     ↓     ↓
    Child Child Child
    Chunk Chunk Chunk

The smaller child chunks can be indexed for retrieval.

When a child chunk matches:

    Query
      ↓
    Child Chunk
      ↓
    Parent Document
      ↓
    Return broader context

---

# 32. Why Parent Document Retrieval?

Suppose:

    Parent:
    "Complete Employee Leave Policy"

Child chunks:

    Child 1 → Eligibility
    Child 2 → Annual Leave
    Child 3 → Sick Leave
    Child 4 → Holidays

The query:

    "How many annual leave days?"

may retrieve:

    Child 2

But the application can return the broader parent context when needed.

This balances:

    Retrieval Precision
          +
    Context Preservation

---

# 33. Self-Query Retriever

A **Self-Query Retriever** uses an LLM to convert a natural-language query into:

1. A semantic search query
2. Structured metadata filters

Suppose documents contain:

    title
    year
    author
    category

User asks:

    "Find Python tutorials published after 2024."

The system may derive:

    Semantic Query:
    "Python tutorials"

    Filter:
    year > 2024

Then:

    Query
      ↓
    LLM
      ↓
    Search Query + Metadata Filter
      ↓
    Vector Store
      ↓
    Results

---

# 34. Why Self-Query Retrieval is Useful

It is useful when your documents have rich metadata.

For example:

    category = "AI"
    year = 2025
    author = "John"
    language = "Python"

A user can naturally ask:

    "Show me AI documents from 2025."

The Retriever can translate this into a semantic query plus metadata constraints.

---

# 35. Time-Weighted Retriever

A Time-Weighted Retriever considers both:

- Semantic relevance
- Recency

This can be useful when newer information should be preferred.

Example:

    Document A
    Relevance = High
    Age = 3 years

    Document B
    Relevance = Slightly lower
    Age = 2 days

A time-aware strategy may prefer newer information depending on its configuration.

This can be useful for:

- News
- Conversations
- Frequently updated knowledge
- Recent events

---

# 36. Ensemble Retriever

An **Ensemble Retriever** combines multiple retrieval methods.

For example:

    Dense Vector Retriever
             +
    BM25 Keyword Retriever
             ↓
      Ensemble Retriever
             ↓
      Combined Results

This is useful because different retrieval methods have different strengths.

---

# 37. Dense Retrieval

Dense retrieval uses embeddings.

Conceptually:

    Query
      ↓
    Embedding
      ↓
    Vector
      ↓
    Vector Search
      ↓
    Documents

It is good at semantic similarity.

---

# 38. Sparse / Keyword Retrieval

Keyword retrieval focuses more on exact terms.

A common example is:

    BM25

Conceptually:

    Query
      ↓
    Keyword Search
      ↓
    Matching Documents

It can be useful for:

- Product IDs
- Error codes
- Exact names
- Technical terms
- Rare keywords

---

# 39. Hybrid / Ensemble Retrieval

Combining both:

    User Query
         ↓
    ┌────┴─────┐
    ↓          ↓
    Dense      Keyword
    Search     Search
    ↓          ↓
    Results    Results
    └────┬─────┘
         ↓
    Combine / Rank
         ↓
    Final Documents

This can improve retrieval robustness.

---

# 40. Multi-Vector Retriever

A Multi-Vector Retriever can represent a document using multiple vectors.

For example:

    Document
       ↓
    ┌────┼────┐
    ↓    ↓    ↓
    Summary  Chunks  Other Representations
       ↓
    Multiple Vectors

This can be useful when a single embedding does not adequately represent all useful aspects of a document.

---

# 41. Retriever Search Pipeline

A typical vector-based retrieval process is:

    User Query
        ↓
    Query Embedding
        ↓
    Vector Search
        ↓
    Candidate Documents
        ↓
    Ranking / Filtering
        ↓
    Top-K Documents
        ↓
    LLM

The exact stages depend on the Retriever implementation.

---

# 42. Retriever and Metadata

Retrievers can work with document metadata.

Example:

    Document(
        page_content="Annual leave policy...",
        metadata={
            "source": "hr_policy.pdf",
            "page": 10,
            "category": "HR"
        }
    )

Metadata can help with:

- Filtering
- Source attribution
- Citations
- Access control
- Debugging

---

# 43. Retriever and Metadata Filtering

Suppose the knowledge base contains:

    Python
    SQL
    LangChain
    Power BI

A metadata filter can restrict retrieval.

Conceptually:

    Query
      ↓
    Metadata Filter
      ↓
    Vector Search
      ↓
    Relevant Documents

For example:

    category = "LangChain"

The exact filtering syntax depends on the vector store.

---

# 44. Retriever in LCEL

Because a Retriever is Runnable-compatible, it can be composed using LCEL.

Example:

    retrieval_chain = (
        {
            "context": retriever,
            "question": RunnablePassthrough()
        }
        | prompt
        | model
        | parser
    )

Architecture:

    Question
       │
       ├──→ Retriever → Context
       │
       └──→ Passthrough → Question
                              ↓
                           Prompt
                              ↓
                             Model
                              ↓
                            Parser
                              ↓
                            Answer

This is a classic RAG architecture.

---

# 45. Retriever as a RAG Component

The responsibilities can be summarized as:

    Document Loader
    → Load information

    Text Splitter
    → Create chunks

    Embedding Model
    → Create vectors

    Vector Store
    → Store/search vectors

    Retriever
    → Retrieve relevant documents

    LLM
    → Generate answer

---

# 46. Retriever vs Search Engine

A traditional search engine often focuses heavily on lexical matching.

A Retriever can use:

- Vector search
- Keyword search
- Metadata filtering
- Hybrid search
- Multi-query search
- Compression
- Parent-child retrieval
- Other retrieval strategies

Therefore, a Retriever is a broader application-level abstraction.

---

# 47. Retrieval vs Generation

These are separate stages.

### Retrieval

Find information.

    Query
      ↓
    Retriever
      ↓
    Documents

### Generation

Generate an answer.

    Query + Documents
          ↓
         LLM
          ↓
        Answer

Complete:

    Query
      ↓
    Retrieve
      ↓
    Context
      ↓
    Generate
      ↓
    Answer

---

# 48. Retrieval-Augmented Generation

RAG literally means:

    Retrieval
       +
    Augmented
       +
    Generation

### Retrieval

Find relevant information.

### Augmented

Add the retrieved information to the model's context.

### Generation

Use the LLM to generate the final response.

Architecture:

    User Question
          ↓
       Retriever
          ↓
    Relevant Context
          ↓
    Question + Context
          ↓
        LLM
          ↓
       Answer

---

# 49. Retrieval Quality

Retriever quality strongly affects RAG quality.

If retrieval is poor:

    Poor Retrieval
          ↓
    Wrong Context
          ↓
    Poor LLM Answer

Even a powerful LLM cannot reliably answer from context that was never retrieved.

Therefore:

> RAG quality depends heavily on retrieval quality.

---

# 50. Precision and Recall

Two important retrieval concepts are:

## Precision

How many retrieved documents are actually relevant?

High precision:

    Retrieved:
    Relevant
    Relevant
    Relevant

Low precision:

    Retrieved:
    Relevant
    Irrelevant
    Irrelevant
    Irrelevant

---

## Recall

How much of the relevant information was successfully retrieved?

High recall:

    Most relevant information
    ↓
    Successfully retrieved

Low recall:

    Important information
    ↓
    Missed

---

# 51. Precision vs Recall Trade-Off

Suppose:

    k = 2

You may get:

    Very relevant results
    ↓
    High precision
    ↓
    But potentially miss useful information
    ↓
    Lower recall

If:

    k = 20

you may retrieve more relevant information:

    Higher recall

but also:

    More irrelevant information
    ↓
    Lower precision

Therefore, `k` needs to be tuned.

---

# 52. Retriever Configuration

A Retriever may have configuration such as:

    search_type
    search_kwargs
    k
    score_threshold
    filters

Example:

    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 5
        }
    )

Another example:

    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 5,
            "fetch_k": 20
        }
    )

The available options vary by backend.

---

# 53. Retriever Invocation

Because a Retriever is Runnable-compatible:

    documents = retriever.invoke(
        "What are Runnables?"
    )

The result is generally:

    [
        Document(...),
        Document(...),
        Document(...)
    ]

You can inspect:

    documents[0].page_content

and:

    documents[0].metadata

---

# 54. Async Retrieval

Retriever operations can also participate in asynchronous Runnable execution.

Conceptually:

    documents = await retriever.ainvoke(
        query
    )

This is useful in asynchronous applications.

---

# 55. Retriever Batch Processing

Runnable-compatible retrievers can also support batch operations depending on the implementation.

Conceptually:

    results = retriever.batch(
        [
            "What is LangChain?",
            "What is RAG?",
            "What is a Retriever?"
        ]
    )

This is useful for:

- Evaluation
- Batch queries
- Retrieval testing
- Dataset processing

---

# 56. Debugging a Retriever

Always inspect retrieval results before blaming the LLM.

Example:

    documents = retriever.invoke(
        "What is chunk overlap?"
    )

    for i, doc in enumerate(documents):
        print(f"Result {i}")
        print(doc.page_content)
        print(doc.metadata)
        print("-" * 50)

Check:

- Are the results relevant?
- Are important chunks missing?
- Are many results duplicates?
- Is metadata correct?
- Is the query being interpreted correctly?

---

# 57. Retriever Debugging Workflow

Use:

    Query
      ↓
    Retriever
      ↓
    Inspect Results
      ↓
    Are results relevant?
       / \
     Yes  No
      ↓    ↓
     LLM  Improve
           Retrieval

If retrieval is poor, investigate:

- Chunking
- Embeddings
- Vector store
- Search type
- Metadata filters
- Query formulation
- Number of retrieved documents

---

# 58. Common Retriever Problems

## Problem 1: Wrong documents retrieved

Possible causes:

- Poor chunking
- Poor embeddings
- Wrong search configuration
- Bad metadata filtering
- Query ambiguity

---

## Problem 2: Too many duplicate chunks

Possible causes:

- Excessive overlap
- Similarity search returning near-duplicates

Possible solution:

- Try MMR
- Reduce overlap
- Improve chunking

---

## Problem 3: Relevant information is missing

Possible causes:

- `k` is too small
- Poor chunking
- Query wording mismatch

Possible solutions:

- Increase `k`
- Use Multi-Query retrieval
- Improve chunking
- Use better embeddings

---

## Problem 4: Too much irrelevant context

Possible causes:

- `k` is too large
- Weak retrieval threshold
- Poor embeddings

Possible solutions:

- Reduce `k`
- Use score thresholding
- Use contextual compression

---

# 59. Retriever Optimization

To improve retrieval quality, investigate the entire pipeline:

    Document Quality
          ↓
    Chunking
          ↓
    Embeddings
          ↓
    Vector Store
          ↓
    Retriever
          ↓
    Ranking
          ↓
    Retrieved Context

Do not assume that changing only the Retriever will fix every retrieval problem.

---

# 60. Retriever Selection Guide

Use this general decision process:

    What problem are you facing?
             ↓
    ┌────────┼───────────────┬───────────────┐
    ↓        ↓               ↓               ↓
   Basic   Duplicate      Query wording   Metadata
   search  results        problem         filtering
    ↓        ↓               ↓               ↓
 Similarity MMR         Multi-Query      Self-Query

For large documents:

    Parent Document Retriever

For excessive retrieved text:

    Contextual Compression

For keyword + semantic search:

    Ensemble / Hybrid Retrieval

For time-sensitive knowledge:

    Time-Weighted Retrieval

---

# 61. Retriever Comparison

| Retriever | Main Purpose |
|---|---|
| Vector Store Retriever | Basic vector-based retrieval |
| Similarity Retriever | Retrieve most similar documents |
| MMR Retriever | Relevance + diversity |
| Multi-Query Retriever | Generate multiple search queries |
| Contextual Compression | Remove irrelevant retrieved content |
| Parent Document Retriever | Retrieve precise child chunks with broader parent context |
| Self-Query Retriever | Query + metadata filtering |
| Time-Weighted Retriever | Relevance + recency |
| Ensemble Retriever | Combine multiple retrieval methods |
| BM25 Retriever | Keyword/lexical retrieval |
| Multi-Vector Retriever | Represent documents with multiple vectors |

---

# 62. Retriever vs Vector Store vs Embedding Model

This distinction is essential.

## Embedding Model

Converts:

    Text
      ↓
    Vector

## Vector Store

Stores:

    Vectors
      +
    Documents
      +
    Metadata

and performs vector search.

## Retriever

Uses a retrieval strategy to return relevant Documents.

Complete:

    Text
      ↓
    Embedding Model
      ↓
    Vector
      ↓
    Vector Store
      ↓
    Retriever
      ↓
    Relevant Documents

---

# 63. Retriever vs LLM

A Retriever does:

    Find information

An LLM does:

    Understand + Generate

Complete RAG:

    User Question
          ↓
       Retriever
          ↓
       Context
          ↓
         LLM
          ↓
       Answer

---

# 64. Retriever vs Text Splitter

A Text Splitter works before retrieval.

    Document
       ↓
    Text Splitter
       ↓
    Chunks
       ↓
    Embeddings
       ↓
    Vector Store
       ↓
    Retriever

Therefore:

    Text Splitter
    → Creates retrieval units

    Retriever
    → Finds retrieval units

---

# 65. Retriever and Metadata Example

Suppose:

    documents = [
        Document(
            page_content="Python decorators...",
            metadata={
                "category": "Python",
                "year": 2025
            }
        ),
        Document(
            page_content="SQL joins...",
            metadata={
                "category": "SQL",
                "year": 2025
            }
        )
    ]

A metadata-aware retrieval system can potentially restrict search to:

    category = "Python"

This reduces irrelevant retrieval.

---

# 66. Advanced RAG Retrieval Architecture

A more advanced system can look like:

    User Query
         ↓
    Query Analysis
         ↓
    ┌────┴───────────────┐
    ↓                    ↓
    Query Rewrite     Metadata Filter
    ↓                    ↓
    └─────────┬──────────┘
              ↓
       Multiple Retrievers
              ↓
       ┌──────┼──────┐
       ↓      ↓      ↓
     Dense   BM25   Other
       ↓      ↓      ↓
       └──────┼──────┘
              ↓
         Result Fusion
              ↓
            Rerank
              ↓
        Compression
              ↓
      Final Context
              ↓
             LLM
              ↓
           Answer

This is a more advanced retrieval architecture.

---

# 67. Reranking

Initial retrieval may return many candidate documents.

A reranker can then reorder them according to query relevance.

Conceptually:

    Query
      ↓
    Initial Retriever
      ↓
    20 Candidates
      ↓
    Reranker
      ↓
    Top 5
      ↓
    LLM

This can improve retrieval quality when the initial vector search is not precise enough.

---

# 68. Retrieval Pipeline with Reranking

    User Query
         ↓
    Retriever
         ↓
    Candidate Documents
         ↓
       Reranker
         ↓
    Most Relevant Documents
         ↓
        Prompt
         ↓
         LLM
         ↓
       Answer

Reranking is separate from the basic retrieval operation.

---

# 69. Retrieval Failure Handling

A good RAG system should consider:

    What if nothing relevant is found?

Possible strategies include:

- Return "I don't know"
- Ask the user to clarify
- Search using alternative queries
- Use another retriever
- Fall back to a broader search
- Use an external knowledge source

The system should avoid confidently generating an answer from irrelevant context.

---

# 70. Hallucination and Retrieval

Retrieval can reduce hallucination by providing relevant external context.

But:

> Retrieval does not automatically eliminate hallucinations.

Example:

    Poor Retrieval
         ↓
    Wrong Context
         ↓
    LLM
         ↓
    Wrong Answer

Therefore, good RAG requires:

    Good Retrieval
         +
    Good Prompting
         +
    Good Generation
         +
    Proper Evaluation

---

# 71. Grounded Generation

A strong RAG design encourages the LLM to answer using retrieved evidence.

Conceptually:

    User Question
         +
    Retrieved Documents
         ↓
        Prompt
         ↓
         LLM
         ↓
    Grounded Answer

The prompt can instruct the model to avoid making unsupported claims.

---

# 72. Citation Support

Because retrieved Documents often contain metadata, a RAG system can provide source information.

Example:

    Answer:
    Employees receive 20 days of annual leave.

    Source:
    employee_handbook.pdf
    Page: 12

The exact citation implementation depends on the application.

---

# 73. Production Retriever Considerations

When building a production RAG system, consider:

### Retrieval quality

Are the right documents retrieved?

### Latency

How quickly can results be returned?

### Scale

How many documents/chunks exist?

### Filtering

Can access and metadata restrictions be applied?

### Freshness

How quickly are new documents indexed?

### Security

Can users retrieve only documents they are authorized to access?

### Cost

How expensive are embeddings, retrieval, reranking, and LLM calls?

---

# 74. Access Control

This is extremely important for enterprise RAG.

Suppose:

    Employee A
    ↓
    Authorized documents

    Employee B
    ↓
    Different authorized documents

The Retriever must not return documents that the user is not authorized to access.

Conceptually:

    User Identity
         ↓
    Authorization Filter
         ↓
    Retriever
         ↓
    Allowed Documents

Access control should not rely solely on the LLM.

---

# 75. Freshness

Knowledge bases change.

Example:

    Old Policy
        ↓
    New Policy

If the old document remains in the Vector Store, the Retriever may return outdated information.

Therefore, production systems should support:

- Document versioning
- Updates
- Deletions
- Re-indexing
- Metadata-based freshness
- Source synchronization

---

# 76. Retriever Evaluation

A Retriever should be evaluated independently from the LLM.

Important questions:

1. Did it retrieve the correct document?
2. Did it retrieve enough context?
3. Did it retrieve irrelevant information?
4. Did it retrieve duplicate information?
5. Did it respect metadata filters?

Useful retrieval metrics include:

- Precision
- Recall
- Hit Rate
- MRR
- NDCG

---

# 77. MRR

**MRR** stands for:

> Mean Reciprocal Rank

It measures how high the first relevant result appears.

Example:

    Relevant document = Result 1

    Reciprocal Rank = 1 / 1 = 1

If:

    Relevant document = Result 5

then:

    Reciprocal Rank = 1 / 5 = 0.2

Higher MRR generally means relevant results appear earlier.

---

# 78. NDCG

**NDCG** stands for:

> Normalized Discounted Cumulative Gain

It evaluates the ranking quality of retrieved results while considering the position of relevant documents.

Higher-ranked relevant results contribute more strongly than lower-ranked relevant results.

It is useful when evaluating ranked retrieval systems.

---

# 79. Retriever Testing

Create a test dataset:

    Question
       +
    Expected Relevant Documents

Example:

    Q1:
    "What is chunk overlap?"

    Expected:
    Text Splitter documentation

Then test:

    Query
      ↓
    Retriever
      ↓
    Retrieved Documents
      ↓
    Compare with expected documents

This helps objectively measure retrieval quality.

---

# 80. Common Mistakes

## Mistake 1: Confusing Retriever with Vector Store

Remember:

    Vector Store → stores/searches vectors

    Retriever → retrieves relevant Documents

---

## Mistake 2: Assuming similarity search is always enough

Similarity search is a good starting point, but some applications benefit from:

- MMR
- Multi-Query
- Hybrid search
- Reranking
- Compression
- Metadata filtering

---

## Mistake 3: Retrieving too many documents

More context is not automatically better.

Too much context can introduce noise.

---

## Mistake 4: Retrieving too few documents

Too little context can cause missing information.

---

## Mistake 5: Ignoring metadata

Metadata can be important for:

- Filtering
- Security
- Citations
- Versioning

---

## Mistake 6: Blaming the LLM for bad retrieval

Always inspect the retrieved documents first.

If the correct information was never retrieved, the LLM cannot use it.

---

# 81. Best Practices

### 1. Start with similarity retrieval

Use a simple vector-based Retriever first.

### 2. Tune `k`

Evaluate different values based on your data.

### 3. Inspect retrieved documents

Always examine actual retrieval results.

### 4. Use MMR when duplicates are a problem

MMR can increase diversity.

### 5. Use Multi-Query when wording is a problem

Generate alternative query formulations.

### 6. Use metadata filters

Restrict retrieval when useful.

### 7. Consider reranking

For demanding applications, reranking can improve result ordering.

### 8. Consider hybrid retrieval

Combine dense and keyword search when exact terms matter.

### 9. Evaluate independently

Measure retrieval quality separately from answer quality.

### 10. Protect sensitive data

Apply authorization before returning documents to the LLM.

---

# 82. Simple RAG Example

A simplified RAG system can look like:

    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser

    retriever = vector_store.as_retriever(
        search_kwargs={
            "k": 3
        }
    )

    chain = (
        {
            "context": retriever,
            "question": RunnablePassthrough()
        }
        | prompt
        | model
        | StrOutputParser()
    )

    answer = chain.invoke(
        "What is a Runnable?"
    )

The architecture is:

    Question
       │
       ├──→ Retriever → Relevant Context
       │
       └──→ Passthrough → Original Question
                              ↓
                           Prompt
                              ↓
                            Model
                              ↓
                           Parser
                              ↓
                           Answer

---

# 83. Retriever Decision Table

| Problem | Possible Approach |
|---|---|
| Basic semantic retrieval | Similarity Retriever |
| Duplicate results | MMR |
| Query wording varies | Multi-Query Retriever |
| Too much irrelevant content | Contextual Compression |
| Need broader context | Parent Document Retriever |
| Natural-language metadata filters | Self-Query Retriever |
| Need keyword + semantic search | Ensemble / Hybrid Retriever |
| Need recent information | Time-Weighted Retriever |
| Need multiple representations | Multi-Vector Retriever |

---

# 84. Complete LangChain RAG Mental Model

    ┌──────────────────────────────────────┐
    │              INGESTION              │
    │                                      │
    │ Documents                            │
    │    ↓                                 │
    │ Document Loader                      │
    │    ↓                                 │
    │ Documents                            │
    │    ↓                                 │
    │ Text Splitter                        │
    │    ↓                                 │
    │ Chunks                               │
    │    ↓                                 │
    │ Embedding Model                      │
    │    ↓                                 │
    │ Vectors                              │
    │    ↓                                 │
    │ Vector Store                         │
    └──────────────────┬───────────────────┘
                       │
                       ↓
    ┌──────────────────────────────────────┐
    │             RETRIEVAL                │
    │                                      │
    │ User Query                           │
    │    ↓                                 │
    │ Retriever                            │
    │    ↓                                 │
    │ Relevant Documents                   │
    │    ↓                                 │
    │ Context                              │
    └──────────────────┬───────────────────┘
                       │
                       ↓
    ┌──────────────────────────────────────┐
    │             GENERATION               │
    │                                      │
    │ Query + Context                      │
    │    ↓                                 │
    │ Prompt                               │
    │    ↓                                 │
    │ LLM                                  │
    │    ↓                                 │
    │ Answer                               │
    └──────────────────────────────────────┘

---

# 85. The Most Important Distinction

Memorize this:

    Document Loader
    → LOAD

    Text Splitter
    → SPLIT

    Embedding Model
    → EMBED

    Vector Store
    → STORE + SEARCH

    Retriever
    → RETRIEVE

    LLM
    → GENERATE

Complete:

    LOAD
      ↓
    SPLIT
      ↓
    EMBED
      ↓
    STORE
      ↓
    RETRIEVE
      ↓
    AUGMENT
      ↓
    GENERATE

---

# 86. Interview Questions

## Q1. What is a Retriever in LangChain?

A Retriever is a component that takes a query and returns relevant Documents from a knowledge source.

---

## Q2. Why are Retrievers important in RAG?

Retrievers find relevant information that can be provided to an LLM as context, allowing the model to generate answers grounded in external knowledge.

---

## Q3. What is the difference between a Retriever and a Vector Store?

A Vector Store stores and searches vectors, while a Retriever provides a retrieval interface for obtaining relevant Documents.

---

## Q4. Is a Retriever a Runnable?

Yes. LangChain Retrievers implement the Runnable interface and can therefore participate in Runnable and LCEL pipelines.

---

## Q5. What does `as_retriever()` do?

It creates a Retriever interface from a compatible Vector Store.

Example:

    retriever = vector_store.as_retriever()

---

## Q6. What is `k`?

`k` generally specifies how many documents should be returned.

Example:

    k = 5

means retrieve five documents.

---

## Q7. What is MMR?

MMR stands for Maximum Marginal Relevance. It balances query relevance with diversity among retrieved documents.

---

## Q8. What is Multi-Query Retrieval?

It uses an LLM to generate multiple alternative versions of a user's query and retrieves documents for those queries, potentially improving recall.

---

## Q9. What is Contextual Compression?

It compresses retrieved documents to retain the information most relevant to the user's query.

---

## Q10. What is Parent Document Retrieval?

It uses smaller child chunks for precise retrieval while allowing the system to return broader parent-document context.

---

## Q11. What is Self-Query Retrieval?

It uses an LLM to convert a natural-language query into a semantic search query and structured metadata filters.

---

## Q12. What is Ensemble Retrieval?

It combines multiple retrieval methods, such as dense vector retrieval and keyword retrieval.

---

## Q13. Why can retrieval quality affect answer quality?

Because the LLM can only use the context supplied to it. If relevant information is not retrieved, the LLM may not be able to produce the correct answer.

---

## Q14. What is the difference between precision and recall in retrieval?

Precision measures how many retrieved documents are relevant.

Recall measures how much of the relevant information was successfully retrieved.

---

## Q15. Why should we inspect retrieved documents?

To determine whether the problem is caused by retrieval rather than generation.

---

# 87. Quick Revision

## Retriever

    Query
      ↓
    Retriever
      ↓
    Relevant Documents

## Vector Store

    Vectors
      ↓
    Search
      ↓
    Similar Documents

## Similarity Retrieval

    Query
      ↓
    Most Similar Documents

## MMR

    Relevance
       +
    Diversity
       ↓
    Results

## Multi-Query

    One Query
       ↓
    Multiple Queries
       ↓
    Multiple Searches
       ↓
    Combined Results

## Contextual Compression

    Retrieved Documents
       ↓
    Compression
       ↓
    Relevant Content

## Parent Document

    Query
      ↓
    Child Chunk
      ↓
    Parent Document
      ↓
    Broader Context

## Self-Query

    Natural Language Query
           ↓
          LLM
           ↓
    Search Query + Filters
           ↓
       Retrieval

## Ensemble

    Dense Search
         +
    Keyword Search
         ↓
    Combined Results

---

# 88. Final One-Line Definition

> **A Retriever in LangChain is a Runnable-compatible abstraction that takes a user query and retrieves relevant Documents from one or more knowledge sources so that the retrieved information can be used as context for downstream processing, especially RAG.**

---

# 89. Final Mental Model

The easiest way to remember the complete concept is:

    USER QUESTION
          ↓
       RETRIEVER
          ↓
    "Find what matters"
          ↓
    RELEVANT DOCUMENTS
          ↓
       CONTEXT
          ↓
        PROMPT
          ↓
          LLM
          ↓
        ANSWER

And the complete RAG system:

    DATA
      ↓
    Document Loader
      ↓
    Documents
      ↓
    Text Splitter
      ↓
    Chunks
      ↓
    Embedding Model
      ↓
    Vectors
      ↓
    Vector Store
      ↓
    Retriever
      ↓
    Relevant Chunks
      ↓
    Prompt
      ↓
    LLM
      ↓
    Answer

> **The Retriever is the bridge between your stored knowledge and the LLM: it decides which pieces of external information should be brought into the model's context for a particular query.**